# LNP-Transfection: Colab Runner

This notebook runs the full pipeline headlessly on Google Colab, predicting
**cell-specific transfection efficiency** from ionizable lipid nanoparticle (LNP) SMILES strings. It uses ECFP4 Morgan fingerprints and physicochemical descriptors alongside an XGBoost regressor. The pipeline supports **K-fold cross-validation**, **Murcko scaffold splitting**, and **buffered similarity splitting** to ensure robust generalization to unseen chemical spaces. The buffered similarity splitting approach enforces a minimum distance buffer between training and test molecules, providing mathematically guaranteed separation. SHAP (SHapley Additive exPlanations) is integrated to provide feature attribution, revealing how specific substructures and physical properties (such as Molecular Weight and Lipophilicity) drive biological performance.

In [ ]:
# @title 1. Configuration
REPO_URL = "https://github.com/Hich00b/lnp-transfection-ml.git"  # @param {type:"string"}
REPO_DIR = "lnp-transfection-ml"  # @param {type:"string"}
RAW_CSV = "data/raw/dataset.csv"  # @param {type:"string"}
print(f"""Repository: {REPO_URL}
Clone dir:  {REPO_DIR}
Raw CSV:    {RAW_CSV}""")

In [ ]:
# @title 2. Clone the repository
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}

In [ ]:
# @title 3. Install dependencies
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# @title 4. Generate features (Morgan fingerprints + descriptors)
!python src/data_processing.py --input {RAW_CSV} --out-dir data/processed

In [ ]:
# @title 5. Train the XGBoost transfection model (two options)
# Option A: K-fold cross-validation (in-distribution/interpolation)
# Option B: Buffered leave-one-group-out (out-of-distribution/extrapolation) with distance cutoff=0.1
# Uncomment one of the lines below to run your preferred option

# Option A: K-fold cross-validation
# !python src/train.py --features data/processed/features.csv --targets data/processed/targets.csv --cv 5

# Option B: Buffered leave-one-group-out (recommended for out-of-distribution generalization)
!python src/train.py --features data/processed/features.csv --targets data/processed/targets.csv --split-method buffered --distance-cutoff 0.1

In [ ]:
# @title 6. SHAP interpretability analysis (on hold-out test set from buffered split)
!python src/evaluate.py --model models/xgb_transfection.pkl --features data/processed/features.csv

In [ ]:
# @title 7. Display the SHAP summary plot
from IPython.display import Image, display
display(Image("models/shap_summary_transfection.png"))
display(Image("models/shap_descriptors_transfection.png"))

## Outputs

| Artefact | Location |
|---|---|
| Feature matrix | `data/processed/features.csv` |
| Targets | `data/processed/targets.csv` |
| Trained model | `models/xgb_transfection.pkl` |
| SHAP beeswarm | `models/shap_summary_transfection.png` |
| Descriptor importance | `models/shap_descriptors_transfection.png` |

The outputs above are from the buffered similarity split (distance cutoff=0.1) training run shown in cell 5.
Defaults target the `Transfection` column (AGILE format). To predict another
numeric target, pass `--target <column>` to `train.py` and `evaluate.py`.